In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(cowplot)
  library(data.table)
  library(repr)
})
options(repr.plot.width = 12, repr.plot.height = 8)

In [ ]:
projdir <- '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'
outdir  <- paste0(projdir, '/pdf/figure1')
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

# data source: HPRC2 supplementary table S6 (per-individual sequencing data summary) x S3
# (1000G-derived superpopulation secondary labels), exported to CSV by
# scripts/harmonize_hg38/pilot/export_supp_qc_csv.py -- authoritative, published numbers,
# preferred over this project's own P04-computed QC now that this table is in hand (P04 kept
# as a low-priority cross-validation option, not removed -- its HG00126 numbers already
# matched this table closely: read_N50_ont 107245.82 here vs. 107636 from P04)

In [ ]:
### params
axis_size  <- 16
title_size <- 18
pt_size    <- 1.6

### palette -- same hex codes pool_design's own notebook uses for superpopulation color
### (ipynb/vcf_metrics/local/02b_pool_nomination_n8.ipynb sp_colors), kept consistent
### across projects rather than redefined independently
superpop_hexes <- c('AFR' = '#d62728', 'EUR' = '#1f77b4', 'SAS' = '#ff7f0e',
                     'EAS' = '#2ca02c', 'AMR' = '#9467bd')

In [ ]:
### base theme -- defined once rather than copy-pasted per plot (SUMMARY.md lint #2),
### panel/plot background blanked for AI compatibility (SUMMARY.md #10)
base_theme <- theme_bw() +
  theme(
    axis.text.x  = element_text(size = axis_size, color = 'black'),
    axis.text.y  = element_text(size = axis_size, color = 'black'),
    axis.title.x = element_text(size = axis_size),
    axis.title.y = element_text(size = axis_size),
    plot.title   = element_text(size = title_size, hjust = 0.5),
    axis.line    = element_line(color = 'black'),
    panel.border = element_blank(),
    panel.grid   = element_blank(),
    panel.background = element_blank(),
    plot.background  = element_blank(),
    legend.position  = 'none'
  )

In [ ]:
### load data
### na.strings must include "" explicitly -- fread's default (na.strings="NA" only) does NOT
### treat an empty string as NA for character columns, so the 6 samples with no superpopulation
### mapping silently became a 6th, unlabeled factor level instead of being dropped by
### !is.na() below. Caught by table(df_sp$superpopulation) printing 6 counts under 5 labels.
df <- as.data.frame(fread(paste0(projdir, '/results/qc/data/supp_seq_qc.csv'),
                           na.strings = c('', 'NA')))

In [ ]:
head(df)

In [ ]:
dim(df)

In [ ]:
### restrict to samples with a 1000G superpopulation label (drops 6 GIAB/reference/non-1000G
### samples with no S3 mapping -- same gap this project's own docs already noted for HG005 etc.)
df_sp <- df[!is.na(df$superpopulation), ]
table(df_sp$superpopulation)

In [ ]:
### shuffle before plotting so overplotted jitter points aren't ordered by input row
set.seed(1)
df_sp <- df_sp[sample(nrow(df_sp)), ]
df_sp$superpopulation <- factor(df_sp$superpopulation, levels = names(superpop_hexes))

In [ ]:
### one panel-maker, reused per metric (SUMMARY.md lint #2 pattern: avoid copy-pasting the
### full ggplot call per panel)
make_qc_panel <- function(df, y_col, y_lab) {
  ggplot(df, aes(x = superpopulation, y = .data[[y_col]], color = superpopulation)) +
    geom_boxplot(outlier.shape = NA, color = 'black', fill = NA) +
    geom_jitter(width = 0.15, size = pt_size, alpha = 0.8) +
    scale_color_manual(values = superpop_hexes) +
    labs(x = NULL, y = y_lab) +
    base_theme
}

In [ ]:
### 4-panel QC overview: ONT + HiFi coverage and read-length (N50), by superpopulation --
### the comparison this project needs before treating superpopulations as equally well-powered
p_cov_ont  <- make_qc_panel(df_sp, 'coverage_ont',  'ONT coverage (x)')
p_cov_hifi <- make_qc_panel(df_sp, 'coverage_hifi', 'HiFi coverage (x)')
p_n50_ont  <- make_qc_panel(df_sp, 'read_N50_ont',  'ONT read N50 (bp)')
p_n50_hifi <- make_qc_panel(df_sp, 'n50_hifi',       'HiFi read N50 (bp)')

### shared legend extracted once (cowplot pattern, SUMMARY.md)
legend_src <- p_cov_ont + theme(legend.position = 'right')
shared_legend <- get_legend(legend_src)

panel_grid <- plot_grid(p_cov_ont, p_cov_hifi, p_n50_ont, p_n50_hifi,
                         labels = c('A', 'B', 'C', 'D'), nrow = 2)
fig1 <- plot_grid(panel_grid, shared_legend, rel_widths = c(1, 0.12))
fig1

In [ ]:
### save -- cairo_pdf embeds fonts (Adobe Illustrator-safe)
ggsave(paste0(outdir, '/fig_1_qc_by_superpop.pdf'), fig1, width = 10, height = 7.5,
       device = cairo_pdf)

In [ ]:
sessionInfo()
Sys.time()